<h1><font color="#113D68" size=6>Deep Learning con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 6. Redes Neuronales Recurrentes</font></h1>

<h1><font color="#113D68" size=4>6. Práctica: Procesamiento del Lenguaje Natural</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>

* [0. Contexto](#section0)
* [1. Descripción del problema: generación de texto](#section1)
* [2. LSTM de linea base](#section2)
    * [2.1. Cargar el dataset](#section2.1)
    * [2.2. Conversión a numérico](#section2.2)
    * [2.3. Dimensiones del dataset](#section2.3)
    * [2.4. Procesamiento de datos](#section2.4)
    * [2.5. Diseño de la LSTM](#section2.5)
    * [2.6. Crear puntos de control](#section2.6)
    * [2.7. Resultados](#section2.7)
* [3. Generación de texto con LSTM](#section3)
    * [3.1. Cargar los pesos de LSTM](#section3.1)
    * [3.2. Convertir de entero a carácter](#section3.2)
    * [3.3. Resultados y evaluación](#section3.3)
* [4. LSTM más profunda](#section4)
* [5. Mejorar nuestro modelo](#section5)

---
<a id="section0"></a>
# <font color="#004D7F" size=6> 0. Contexto</font>

En este proyecto, aprenderemos a crear un modelo generativo de texto utilizando LSTM:
* Descripción de un modelos generativos de texto.
* Enmarcar el problema de las secuencias de texto a un modelo generativo.
* Desarrollar una LSTM para generar secuencias de texto.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section1"></a>
# <font color="#004D7F" size=6>1. Descripción del problema: generación de texto</font>

VAmos a trabajar el libro de "Alicia en el país de las maravillas", por lo que podemos descargarlo de la página (Texto sin formato UTF-8) de este libro de forma gratuita y colocarlo en su directorio de trabajo con el nombre `wonderland.txt`. 

Abrimos el archivo y eliminamos el encabezado:
```
        *** START OF THIS PROJECT GUTENBERG EBOOK ALICE'S ADVENTURES IN WONDERLAND ***
```

El pie de página es todo el texto después de la línea de texto que dice:
```
        THE END
```

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Más información sobre el [Proyecto Gutenberg](https://www.gutenberg.org/)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Descargar el libro [Alice's Adventures in Wonderland](https://www.gutenberg.org/ebooks/11)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. LSTM de linea base</font>

En esta sección desarrollaremos una red LSTM simple para aprender secuencias de caracteres de Alicia en el país de las maravillas. 

<a id="section2.1"></a>
# <font color="#004D7F" size=5>2.1. Cargar el dataset</font>

Comencemos importando las clases y funciones que pretendemos usar para entrenar nuestro modelo.

Debemos cargar el texto

In [ ]:
# Descarga del libro desde el Proyecto Gutenberg (necesario en Colab) y
# eliminación del encabezado y pie de página
import os, urllib.request

os.makedirs('Datasets', exist_ok=True)
if not os.path.exists('Datasets/wonderland.txt'):
    texto = urllib.request.urlopen('https://www.gutenberg.org/cache/epub/11/pg11.txt').read().decode('utf-8')
    texto = texto.replace('\r\n', '\n')
    inicio = texto.find('\nCHAPTER I.\n') + 1           # inicio del capítulo 1 (tras el índice)
    fin = texto.find('THE END') + len('THE END')
    with open('Datasets/wonderland.txt', 'w', encoding='utf-8') as f:
        f.write(texto[inicio:fin])

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

# cargar el libro y convertirlo a minúsculas (reduce el vocabulario)
filename = "Datasets/wonderland.txt"
raw_text = open(filename, 'r', encoding='utf-8').read()
raw_text = raw_text.lower() 

<a id="section2.2"></a>
# <font color="#004D7F" size=5>2.2. Conversión a numérico</font>

No podemos modelar los caracteres directamente, sino que debemos convertir los caracteres a números enteros. 

Por ejemplo, la lista de caracteres en minúscula ordenados únicos en el libro es la siguiente:

In [ ]:
# mapeo carácter -> entero
chars = sorted(list(set(raw_text)))
char_to_int = dict((c, i) for i, c in enumerate(chars))
print(chars) 

{'\n': 0,
 ' ': 1,
 '!': 2,
 '"': 3,
 '#': 4,
 '$': 5,
 '%': 6,
 "'": 7,
 '(': 8,
 ')': 9,
 '*': 10,
 ',': 11,
 '-': 12,
 '.': 13,
 '/': 14,
 '0': 15,
 '1': 16,
 '2': 17,
 '3': 18,
 '4': 19,
 '5': 20,
 '6': 21,
 '7': 22,
 '8': 23,
 '9': 24,
 ':': 25,
 ';': 26,
 '?': 27,
 '@': 28,
 '[': 29,
 ']': 30,
 '_': 31,
 'a': 32,
 'b': 33,
 'c': 34,
 'd': 35,
 'e': 36,
 'f': 37,
 'g': 38,
 'h': 39,
 'i': 40,
 'j': 41,
 'k': 42,
 'l': 43,
 'm': 44,
 'n': 45,
 'o': 46,
 'p': 47,
 'q': 48,
 'r': 49,
 's': 50,
 't': 51,
 'u': 52,
 'v': 53,
 'w': 54,
 'x': 55,
 'y': 56,
 'z': 57,
 'ù': 58,
 '—': 59,
 '‘': 60,
 '’': 61,
 '“': 62,
 '”': 63,
 '\ufeff': 64}

<a id="section2.3"></a>
# <font color="#004D7F" size=5>2.3. Dimensiones del dataset</font>

Ahora que se cargó el libro y se preparó el mapeo, podemos resumir el conjunto de datos.

In [ ]:
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab) 

Total Characters:  164202
Total Vocab:  65


En este tutorial, dividiremos el texto del libro en subsecuencias con una longitud fija de 100 caracteres, una longitud arbitraria. 

Cada patrón de entrenamiento de la red se compone de 100 pasos de tiempo de un carácter (X) seguidos de una salida de carácter (y). 

Por ejemplo, si la longitud de la secuencia es 5 (para simplificar), los dos primeros patrones de entrenamiento serían los siguientes:
```
            CHAPT -> E
            HAPTE -> R
```
A medida que dividimos el libro en estas secuencias, convertimos los caracteres a números enteros usando nuestra tabla de búsqueda que preparamos anteriormente.

In [ ]:
# ventanas de 100 caracteres -> el carácter siguiente
seq_length = 100
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
    seq_in = raw_text[i:i + seq_length]
    seq_out = raw_text[i + seq_length]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append(char_to_int[seq_out])
n_patterns = len(dataX)
print("Total Patterns: ", n_patterns) 

Total Patterns:  164102


<a id="section2.4"></a>
# <font color="#004D7F" size=5>2.4. Procesamiento de datos</font>

Al ejecutar lo anterior, puedes ver que se crean un total de 144,243 muestras. Cada muestra ahora está en forma de enteros, transformada utilizando la mapeo `char_to_int`. Sin embargo, un modelo de PyTorch prefiere ver los datos en tensores de punto flotante. Por lo tanto, deberías convertirlos en tensores de PyTorch.

1. Primero debemos transformar la lista de secuencias de entrada en la forma [muestras, pasos de tiempo, características] esperada por una red LSTM. 
2.  Para facilitar el entrenamiento, también es una buena idea normalizar la entrada de 0 a 1. Por lo tanto, tienes lo siguiente:

In [ ]:
# [muestras, pasos de tiempo, características] y normalización a [0, 1]
X = torch.tensor(dataX, dtype=torch.float32).reshape(n_patterns, seq_length, 1)
X = X / float(n_vocab) 

y = torch.tensor(dataY)  # índices de clase para CrossEntropyLoss
print(X.shape, y.shape) 

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], dtype=float32)

<a id="section2.5"></a>
# <font color="#004D7F" size=5>2.5. Diseño de la LSTM</font>

Ahora podemos definir nuestro modelo LSTM. 
1. Definimos una única capa LSTM oculta con 256 unidades de memoria. 
2. La red utiliza un Dropout del 20%. 
3. La capa de salida es una capa completamente conectada para producir logits para los 44 vocablos. Los logits pueden convertirse en predicciones tipo probabilidad usando una función softmax.

In [ ]:
class CharModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=256, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.linear = nn.Linear(256, n_vocab)
    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]  # solo la salida del último paso de tiempo
        x = self.linear(self.dropout(x))  # logits sobre el vocabulario
        return x 

<a id="section2.6"></a>
# <font color="#004D7F" size=5>2.6. Proceso de entrenamiento</font>

Este es un modelo para la clasificación de un solo carácter en 44 clases. 
- Utilizar la pérdida de entropía cruzada. 
- Se optimiza usando el optimizador Adam. 
- Para simplificar, no se ha creado un conjunto de pruebas, pero el modelo se evalúa nuevamente con el conjunto de entrenamiento al final de cada época para seguir el progreso.
- Para preservar el resultado del trabajo, se guarda el mejor modelo encontrado para su reutilización futura.

In [ ]:
import copy

# usa GPU si está disponible (en Colab: Entorno de ejecución -> Cambiar tipo -> GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n_epochs = 40
batch_size = 128
model = CharModel().to(device)

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss(reduction="sum")
loader = data.DataLoader(data.TensorDataset(X, y), shuffle=True, batch_size=batch_size)

best_model = None
best_loss = np.inf
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch.to(device))
        loss = loss_fn(y_pred, y_batch.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    # validación (sobre el propio train, por simplicidad)
    model.eval()
    loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred = model(X_batch.to(device))
            loss += loss_fn(y_pred, y_batch.to(device))
        if loss < best_loss:
            best_loss = loss
            best_model = copy.deepcopy(model.state_dict())
        print("Epoch %d: Cross-entropy: %.4f" % (epoch, loss))

# se guardan los pesos y el mapeo de caracteres juntos
torch.save([best_model, char_to_int], "single-char.pth") 

Al completar el ciclo de entrenamiento, deberías tener el archivo `single-char.pth` creado, que contiene el mejor peso del modelo encontrado, así como el mapeo de carácter a entero utilizado por este modelo.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. Generación de texto con LSTM</font>

La generación de texto utilizando la red LSTM entrenada es relativamente sencilla. 

<a id="section3.1"></a>
# <font color="#004D7F" size=5>3.1. Cargar los pesos de LSTM</font>

En primer lugar, cargamos el objeto guardado

In [ ]:
best_model, char_to_int = torch.load("single-char.pth")
n_vocab = len(char_to_int)
int_to_char = dict((i, c) for c, i in char_to_int.items()) 

<a id="section3.2"></a>
# <font color="#004D7F" size=5>3.2. Recargar el modelo</font>

Recargamos el modelo indicado

In [ ]:
class CharModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=256, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.linear = nn.Linear(256, n_vocab)
    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.linear(self.dropout(x))
        return x
    
model = CharModel()
model.load_state_dict(best_model)

model.to(device)

<a id="section3.3"></a>
# <font color="#004D7F" size=5>3.3. Resultados y evaluación</font>

Finalmente, necesitamos realmente hacer predicciones de manera que:
1. Comenzamos primero con una semilla como entrada
2. Generamos el siguiente carácter y 
3. Actualizamos la semilla para agregar el carácter generado al final y recortar el primer carácter. 

Este proceso se repite mientras queramos predecir nuevos caracteres (por ejemplo, una secuencia de 100 caracteres de longitud). 

In [ ]:
filename = "Datasets/wonderland.txt"
seq_length = 100
raw_text = open(filename, 'r', encoding='utf-8').read()
raw_text = raw_text.lower()
start = np.random.randint(0, len(raw_text) - seq_length)
prompt = raw_text[start:start + seq_length]  # semilla aleatoria del libro
pattern = [char_to_int[c] for c in prompt]

model.eval()
print('Prompt: "%s"' % prompt)
print()
with torch.no_grad():
    for i in range(1000):
        x = np.reshape(pattern, (1, len(pattern), 1)) / float(n_vocab)
        x = torch.tensor(x, dtype=torch.float32)
    
        prediction = model(x.to(device))
    
        index = int(prediction.argmax())  # carácter más probable
        result = int_to_char[index]
        print(result, end="")
        
        pattern.append(index)
        pattern = pattern[1:]
print()
print("Done.")

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. LSTM más profunda</font>

Ahora, podemos intentar mejorar la calidad del texto generado creando una red mucho más profunda. 

El único cambio realizado es que se ha añadido `num_layers=2`.

In [ ]:
class CharModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=256, num_layers=2, batch_first=True, dropout=0.2)  # 2 capas apiladas
        self.dropout = nn.Dropout(0.2)
        self.linear = nn.Linear(256, n_vocab)
    def forward(self, x):
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.linear(self.dropout(x))
        return x

Añadimos todo el código restante.

In [ ]:
import copy
n_epochs = 40
batch_size = 128
model = CharModel()

optimizer = optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss(reduction="sum")
loader = data.DataLoader(data.TensorDataset(X, y), shuffle=True, batch_size=batch_size)

best_model = None
best_loss = np.inf
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in loader:
        y_pred = model(X_batch.to(device))
        loss = loss_fn(y_pred, y_batch.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    loss = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred = model(X_batch.to(device))
            loss += loss_fn(y_pred, y_batch.to(device))
        if loss < best_loss:
            best_loss = loss
            best_model = copy.deepcopy(model.state_dict())  # copia: state_dict() devuelve referencias
        print("Epoch %d: Cross-entropy: %.4f" % (epoch, loss))

torch.save([best_model, char_to_int], "single-char.pth")

In [ ]:

best_model, char_to_int = torch.load("single-char.pth")
n_vocab = len(char_to_int)
int_to_char = dict((i, c) for c, i in char_to_int.items())
model.load_state_dict(best_model)

filename = "Datasets/wonderland.txt"
seq_length = 100
raw_text = open(filename, 'r', encoding='utf-8').read()
raw_text = raw_text.lower()
start = np.random.randint(0, len(raw_text)-seq_length)
prompt = raw_text[start:start+seq_length]
pattern = [char_to_int[c] for c in prompt]

model.eval()
print('Prompt: "%s"' % prompt)
with torch.no_grad():
    for i in range(1000):
        x = np.reshape(pattern, (1, len(pattern), 1)) / float(n_vocab)
        x = torch.tensor(x, dtype=torch.float32)
        prediction = model(x.to(device))
        index = int(prediction.argmax())
        result = int_to_char[index]
        print(result, end="")
        pattern.append(index)
        pattern = pattern[1:]
print()
print("Done.")



---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section5"></a>
# <font color="#004D7F" size=6>5. Mejorar nuestro modelo</font>

A continuación, se muestra una muestra de ideas que tal vez desee investigar para mejorar aún más el modelo:
* Predecir menos de 1000 caracteres como salida para una semilla determinada.
* Eliminar toda la puntuación del texto fuente y, por tanto, del vocabulario de los modelos.
* Pruebe un One-Hot Encoding para las secuencias de entrada.
* Entrene al modelo en oraciones rellenas en lugar de secuencias aleatorias de caracteres.
* Aumentar el número de épocas de entrenamiento a 100 o más.
* Agregue Dropout a la capa de entrada visible y considere ajustar el porcentaje de Dropout.
* Ajuste el tamaño de batch, pruebe con un tamaño de batch de 1 como línea de base (muy lenta) y tamaños más grandes a partir de ahí.
* Agregue más unidades de memoria a las capas y / o más capas.
* Cambie las capas de LSTM para que tengan estado para mantener el estado en todos los batch.

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>